In [1]:
zones = {
    "takeoff": ["W1", "W2", "W3", "W4"],
    "mid":     ["W5", "W6", "W7", "W8"],
    "landing": ["W9", "W10", "W11", "W12"]
}

wind_features = ["Speed", "Tangent", "Cross", "Turbulence"]

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

def plot(X_seq, X_sim, j, name, error):
    """
    Plot actual vs simulated trajectory in 3D with projections.
    Args:
        X_seq (ndarray): ground truth state sequence (T, state_dim)
        X_sim (ndarray): simulated state sequence (T, state_dim)
        j (int): index of jump
        name (str): identifier for saving plots """

    save_dir = f"plots/{name}"
    os.makedirs(save_dir, exist_ok=True)

    
    actual = X_seq[:, :3]   #vzame X,Y,Z iz [x, y, z, vx, vy, vz]
    sim = X_sim[:, :3]          # same iz simulacije

    Xa, Ya, Za = actual[:,0], actual[:,1], actual[:,2]
    Xs, Ys, Zs = sim[:,0], sim[:,1], sim[:,2]

    #napaka = np.mean(error_fun(actual, sim))
    napaka = error
    length = (Xs[-1]**2 + Ys[-1]**2 + Zs[-1]**2)**(1/2)
    length_a = (Xa[-1]**2 + Ya[-1]**2 + Za[-1]**2)**(1/2)

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot3D(Xa, Ya, Za, label="Actual", color="blue")
    ax.plot3D(Xs, Ys, Zs, label="Simulated", color="red", linestyle="--")


    # Ground projections
    ax.plot(Xa, 15, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(Xs, 15, Zs, color="red", alpha=0.3, linestyle=':')

    ax.plot(0, Ya, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(0, Ys, Zs, color="red", alpha=0.3, linestyle=':')


    ax.set_title(f"Actual vs Simulated Trajectory\n Error: {napaka:.3f}, Simulated length: {length:.1f},  Actual length: {length_a:.1f}")
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.legend()
    ax.set_box_aspect([1,1,1])
    ax.set_ylim(-30, 30)

    plt.legend()
    #plt.tight_layout()

    filename = f"2SSM flight_simulation{j + 1}.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close(fig)
    #plt.show()


### old functions

In [3]:
import numpy as np
from scipy.interpolate import interp1d

def resample_curve(curve, n_points=200):
    """
    Resample a 3D curve to have exactly n_points, parameterized by arc length.

    Args:
        curve: (N, d) array, trajectory points
        n_points: number of resampled points

    Returns:
        (n_points, d) array, resampled curve
    """
    # compute arc length
    diffs = np.diff(curve, axis=0)
    seg_lengths = np.linalg.norm(diffs, axis=1)
    arc = np.concatenate([[0], np.cumsum(seg_lengths)])

    # normalize arc length to [0, 1]
    arc_norm = arc / arc[-1]

    # create interpolators
    resampled = []
    t_new = np.linspace(0, 1, n_points)
    for d in range(curve.shape[1]):
        f = interp1d(arc_norm, curve[:, d], kind="linear")
        resampled.append(f(t_new))
    return np.stack(resampled, axis=1)

In [16]:
def curve_error_with_overshoot(curve_a, curve_b, n_points=200):
    # resample both
    len_a = len(curve_a)
    len_b = len(curve_b)

    n_common = min(len_a, len_b)
    a_res = resample_curve(curve_a, n_common)
    b_res = resample_curve(curve_b, n_common)

    # pointwise error on common part
    errors = list(np.linalg.norm(a_res - b_res, axis=1))

    # overshoot penalty
    if len_a > len_b:
        last_b = curve_b[-1]
        extra = curve_a[len_b:]
        for p in extra:
            errors.append(np.linalg.norm(p - last_b))
    elif len_b > len_a:
        last_a = curve_a[-1]
        extra = curve_b[len_a:]
        for p in extra:
            errors.append(np.linalg.norm(p - last_a))

    return np.mean(errors), np.array(errors)


### new funtions

In [3]:
def interpolate_curve_by_x(curve):
    """
    Interpolates a 3D curve by x-coordinate and returns (x, y, z),
    keeping the original start and end points exactly.

    Args:
        curve : array-like of shape (N, 3)
            Each row is [x, y, z].

    Returns:
        result : ndarray of shape (M, 3)
            Interpolated coordinates at integer x plus original endpoints.
    """

    curve = np.asarray(curve, dtype=float)
    if curve.ndim != 2 or curve.shape[1] != 3:
        raise ValueError("sth")

    curve = curve[np.argsort(curve[:, 0])]

    x = curve[:, 0]
    y = curve[:, 1]
    z = curve[:, 2]    

    x_new = np.arange(np.ceil(x.min()) + 0, np.floor(x.max()) + 1)

    f_y = interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate")
    f_z = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate")

    if len(x_new) > 0:
        y_new = f_y(x_new)
        z_new = f_z(x_new)
        result = np.vstack([
            curve[0],
            np.column_stack((x_new, y_new, z_new)),
            curve[-1]
        ])
    else:
        result = curve

    return result


In [4]:
def error_fun(curveA, curveB):
    curve1 = interpolate_curve_by_x(curveA)
    curve2 = interpolate_curve_by_x(curveB)

    if len(curve1) < len(curve2):
        longer = curve2
        shorter = curve1
    else:
        longer = curve1
        shorter = curve2

    longer1 = longer[:len(shorter)]
    shorter1 = shorter

    pad_len = len(longer) - len(shorter)
    if pad_len > 0:
        pad = np.repeat(shorter[-1][None, :], pad_len, axis=0)
        shorter2 = pad
        longer2 = longer[len(shorter):]

        diffs = np.vstack([longer1 - shorter1, longer2 - shorter2])
    else:
        diffs = longer1 - shorter1

    dists = np.linalg.norm(diffs, axis=1)
    return np.mean(dists)

In [6]:


def preprocess_flight_normalized(df, step=0.05):
    """
    Preprocess a normalized flight dataframe (already interpolated) to generate states (X), observations (Y), and controls (U).
    
    Args:
        df (pd.DataFrame): Normalized flight dataframe (fixed time step).
        step (float): Time step between rows (used for derivatives).
        
    Returns:
        states (np.ndarray): State matrix [time_steps, state_dim].
        observations (np.ndarray): Observation matrix [time_steps, obs_dim].
        controls (np.ndarray): Control matrix [time_steps, control_dim].
    """

    df.columns = df.columns.str.strip()   #izloči imena stolpcev
    df = df.iloc[1:]
    
    df = df.ffill().bfill()   #back fill za manjkajoče vrednosti
    
    x = df["X [m]"].to_numpy()   #save values
    y = df["Y [m]"].to_numpy()
    z = df["Z [m]"].to_numpy()
    
    dt = step
    #vx = np.gradient(x, dt)   #gradient za hitrost
    vx = df.get("Speed hor. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)
    vy = np.gradient(y, dt)
    #vz = np.gradient(z, dt)
    vz = df.get("Speed ver. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    speed = df.get("speed resulting [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    opening = df.get("Opening Angle [°]", pd.Series([0]*len(x))).to_numpy()
    roll_L = df.get("Roll Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    roll_R = df.get("Roll Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_L = df.get("Yaw Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_R = df.get("Yaw Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    stall_L = df.get("Stalling Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    stall_R = df.get("Stalling Angle Right [°]", pd.Series([0]*len(x))).to_numpy()

    states = np.stack([x, y, z, vx, vy, vz, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)   #zgradimo state vector X

    height = df.get("Height above ground [m]", pd.Series([0]*len(x))).to_numpy()

    observations = np.stack([x, y, z, height], axis=1)   #zgradimo observation vector Y

    zone_feature_avgs = []
    values = []
    cols = []

    #    #all features
    #for feature in wind_features:
    #    for sensor_numb in range(12):
    #        sensor = f"W{sensor_numb + 1}"
    #        cols.append(f"{sensor}_{feature}")
    #        #print(cols)
    #for col in cols:
    #    val = df.get(col, pd.Series([0]*len(x))).to_numpy() 
    #    values.append(val)
    #
    #controls = np.stack(values, axis=1) 

        #zone
    for feature in wind_features:
        for zone, sensors in zones.items():
            cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
            avg_feature = df[cols].mean(axis=1).to_numpy()
            zone_feature_avgs.append(avg_feature)
    
    controls = np.stack(zone_feature_avgs, axis=1)
    
    return states, observations, controls


In [7]:
import os
import pandas as pd
import numpy as np

normalized_folder = r'C:\Users\vsi\Desktop\ijs\smucarski_skoki\project\simulation\2024_03_Planica_12_winds\cleaned\normalized'
combined_output = os.path.join(normalized_folder, "combined_dataset.npz")
file_names = []

X_list, Y_list, U_list = [], [], []

for filename in os.listdir(normalized_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(normalized_folder, filename)
        df = pd.read_csv(file_path)

        X_state, Y_obs, U_ctrl = preprocess_flight_normalized(df)

        X_list.append(X_state)
        Y_list.append(Y_obs)
        U_list.append(U_ctrl)
        file_names.append(filename)

        #if np.any(np.isnan(X_state)):
        #    print(f"NaN values in {filename}")

        print(f"Processed {filename} -> X:{X_state.shape}, Y:{Y_obs.shape}, U:{U_ctrl.shape}")

# Save combined dataset
#np.savez(combined_output, X=np.array(X_list, dtype=object), Y=np.array(Y_list, dtype=object), U=np.array(U_list, dtype=object))
print(f"\nCombined dataset saved to {combined_output}")


Processed 999_999_Jumper_Anon_ANO_1_20240409-102625_C_OfficialResults_cleaned_normalized.csv -> X:(135, 14), Y:(135, 4), U:(135, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102742_C_OfficialResults_cleaned_normalized.csv -> X:(131, 14), Y:(131, 4), U:(131, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102831_C_OfficialResults_cleaned_normalized.csv -> X:(145, 14), Y:(145, 4), U:(145, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102832_C_OfficialResults_cleaned_normalized.csv -> X:(139, 14), Y:(139, 4), U:(139, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102833_C_OfficialResults_cleaned_normalized.csv -> X:(136, 14), Y:(136, 4), U:(136, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102834_C_OfficialResults_cleaned_normalized.csv -> X:(132, 14), Y:(132, 4), U:(132, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102835_C_OfficialResults_cleaned_normalized.csv -> X:(156, 14), Y:(156, 4), U:(156, 12)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102836_C_OfficialResults

In [9]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from scipy.interpolate import interp1d
import os
import matplotlib.pyplot as plt


def fit_ssm_cv(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.
    Uses arc-length interpolation for error computation.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []
    is_nan = []


    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        # --- build training data ---
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T


        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        # --- training error (shape-aligned) ---
        train_errs = []
        for i in train_idx:
            x0 = X_list[i][0]
            U_seq = U_list[i]
            X_seq = X_list[i]
            
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
        
            train_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            train_errs.append(train_err)

        train_errors.append(np.mean(train_errs))


        # --- test error ---
        test_errs = []
        for j in test_idx:
            x0 = X_list[j][0]
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq) - 1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)

            # compute error
            test_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_errs.append(test_err)

            # plot each trajectory
            plot(X_seq[:, :3], X_sim[:, :3], j, "2-SSM-2-base", test_err)

        test_errors.append(np.mean(test_errs))

        print(np.mean(train_errs), np.mean(test_errs))
        is_nan.append(np.isnan(train_errors))



    return np.mean(train_errors), np.mean(test_errors), models, is_nan


In [10]:
avg_train_err, avg_test_err, models, is_nan = fit_ssm_cv(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test_error:", avg_test_err)

1.8735467306052496 0.8717774638570822
1.8743653570900667 1.1564116968630769
1.865556199979204 2.9407617315851122
1.8760455381857195 0.8459432136274665
1.8688886762644044 1.9706466737659432
1.868164357182027 2.2891727229211942
1.8693721526737637 2.059199693194586
1.8747932579376194 0.9254090904679066
1.8742541734203306 1.229839030597064
1.8775850463798307 1.0769962121109986
1.8720711075220318 1.9830045659760271
1.8775151047719068 1.0111528336140472
1.8704109892145144 2.1547409019660155
1.874612608605244 1.02124935703552
1.8667703350415716 2.463967892031622
1.8669035766615094 2.8149459179504706
1.8699031642496808 2.1715720637322455
1.8739710387784256 0.7248910061183605
1.8708812552736755 2.145311414101452
1.8714465116333996 1.6885988194920334
1.8707474297148792 2.0549497023720513
1.8674555295733863 2.5796231098644973
1.8713575290758655 1.8758045672446808
1.8752257732166715 1.2576721714646775
1.8735288761804252 1.1184368889801597
1.860936134623573 3.7124513114871576
1.8715344604946325 1.2

### quad


In [15]:
wind_features = ["Speed", "Tangent", "Cross", "Turbulence", "Speed_quad", "Tangent_quad", "Cross_quad", "Turbulence_quad"]

In [24]:


def preprocess_flight_normalized(df, step=0.05):
    """
    Preprocess a normalized flight dataframe (already interpolated) to generate states (X), observations (Y), and controls (U).
    
    Args:
        df (pd.DataFrame): Normalized flight dataframe (fixed time step).
        step (float): Time step between rows (used for derivatives).
        
    Returns:
        states (np.ndarray): State matrix [time_steps, state_dim].
        observations (np.ndarray): Observation matrix [time_steps, obs_dim].
        controls (np.ndarray): Control matrix [time_steps, control_dim].
    """

    df.columns = df.columns.str.strip()   #izloči imena stolpcev
    df = df.iloc[1:]
    
    df = df.ffill().bfill()   #back fill za manjkajoče vrednosti
    
    x = df["X [m]"].to_numpy()   #save values
    y = df["Y [m]"].to_numpy()
    z = df["Z [m]"].to_numpy()
    
    dt = step
    #vx = np.gradient(x, dt)   #gradient za hitrost
    vx = df.get("Speed hor. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)
    vy = np.gradient(y, dt)
    #vz = np.gradient(z, dt)
    vz = df.get("Speed ver. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    speed = df.get("speed resulting [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    opening = df.get("Opening Angle [°]", pd.Series([0]*len(x))).to_numpy()
    roll_L = df.get("Roll Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    roll_R = df.get("Roll Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_L = df.get("Yaw Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_R = df.get("Yaw Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    stall_L = df.get("Stalling Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    stall_R = df.get("Stalling Angle Right [°]", pd.Series([0]*len(x))).to_numpy()

    states = np.stack([x, y, z, vx, vy, vz, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)   #zgradimo state vector X

    height = df.get("Height above ground [m]", pd.Series([0]*len(x))).to_numpy()

    observations = np.stack([x, y, z, height], axis=1)   #zgradimo observation vector Y

    zone_feature_avgs = []
    values = []
    cols = []

    for feature in wind_features:
        for sensor_numb in range(12):
            sensor = f"W{sensor_numb + 1}"
            cols.append(f"{sensor}_{feature}")
            #print(cols)
    for col in cols:
        val = df.get(col, pd.Series([0]*len(x))).to_numpy() 
        values.append(val)
    
    controls = np.stack(values, axis=1)  # shape: (time_steps, 12)

    #for feature in wind_features:
    #    for zone, sensors in zones.items():
    #        cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
    #        avg_feature = df[cols].mean(axis=1).to_numpy()
    #        zone_feature_avgs.append(avg_feature)
    #
    #controls = np.stack(zone_feature_avgs, axis=1)  # shape: (time_steps, 12)
    
    return states, observations, controls


In [25]:
import os
import pandas as pd
import numpy as np

normalized_folder = r'C:\Users\vsi\Desktop\ijs\smucarski_skoki\project\simulation\2024_03_Planica_12_winds\cleaned\cleaned_quad\normalized_quad'
combined_output = os.path.join(normalized_folder, "combined_dataset.npz")
file_names = []

X_list, Y_list, U_list = [], [], []

for filename in os.listdir(normalized_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(normalized_folder, filename)
        df = pd.read_csv(file_path)

        X_state, Y_obs, U_ctrl = preprocess_flight_normalized(df)

        X_list.append(X_state)
        Y_list.append(Y_obs)
        U_list.append(U_ctrl)
        file_names.append(filename)

        #if np.any(np.isnan(X_state)):
        #    print(f"NaN values in {filename}")

        print(f"Processed {filename} -> X:{X_state.shape}, Y:{Y_obs.shape}, U:{U_ctrl.shape}")

# Save combined dataset
#np.savez(combined_output, X=np.array(X_list, dtype=object), Y=np.array(Y_list, dtype=object), U=np.array(U_list, dtype=object))
print(f"\nCombined dataset saved to {combined_output}")


Processed 999_999_Jumper_Anon_ANO_1_20240409-102625_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(133, 14), Y:(133, 4), U:(133, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102742_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(129, 14), Y:(129, 4), U:(129, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102831_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(142, 14), Y:(142, 4), U:(142, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102832_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(135, 14), Y:(135, 4), U:(135, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102833_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(132, 14), Y:(132, 4), U:(132, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102834_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(131, 14), Y:(131, 4), U:(131, 96)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102835_C_OfficialResults_cleaned_quad-cleaned_nor

In [26]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

def fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        avg_train_err, avg_test_err, models (list of (A,B,C,D) per fold)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for train_idx, test_idx in kf.split(X_list):
        # --- build training data ---
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)     #ridge is just MNK z regularizacijo
                #
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        control_dim = U_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        # --- training error (shape-aligned) ---
        train_errs = []
        for i in train_idx:
            x0 = X_list[i][0]
            U_seq = U_list[i]
            X_seq = X_list[i]
            
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
        
            train_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            train_errs.append(train_err)

        train_errors.append(np.mean(train_errs))

        # --- compute test error ---
        test_errs = []
        for j in test_idx:
            x0 = X_list[j][0]       #initial state...adjust it
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)

            # compare to ground truth
            test_err = np.mean(np.linalg.norm(X_seq[:, :3] - X_sim[:, :3], axis=1))
            #test_err = penalized_bidirectional_distance(X_seq[::10], X_sim[::10])
            test_errs.append(test_err)

            plot(X_seq, X_sim, j, "2-SSM-quad", test_err)
        test_errors.append(np.mean(test_errs))

        print(np.mean(train_errs), np.mean(test_errs))

    return np.mean(train_errors), np.mean(test_errors), models


In [27]:
avg_train_err, avg_test_err, models = fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test_error:", avg_test_err)

1.6354007367818828 2.4564489841863506
1.6418391963193808 1.826244682439421
1.6402610199485261 4.101658852366884
1.6407716378735815 1.333746626088646
1.6405961326323173 4.149176849998513
1.6354370576161372 5.443755793231163
1.6329233418540874 3.1158271829832387
1.643573576997654 4.0676155122521065
1.6399111918214342 4.316766321654278
1.6425436464054417 1.4718256609676181
1.6347961035876415 6.519205013465885
1.6376031791542918 2.2547212116650073
1.6310064339210724 2.728541072435399
1.6407283462723579 4.619442887623241
1.6361632358592655 3.0179334748803908
1.6360069479248112 5.078289512915446
1.639498635864482 2.7368426689276433
1.639582095657575 6.634540068819609
1.6382901588219083 4.958971734330926
1.6388858928619365 5.939119436987253
1.6356537383321372 5.748931599880127
1.639665441581968 3.830220045666393
1.6399731686017138 4.652937750192189
1.6371337888143713 3.1707236800858913
1.6355549263536027 7.7550844253344655
1.6263057620818777 4.215416589521463
1.6327350863169818 4.133255728233